In [ ]:
import os
# Suppress TensorFlow logging messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
import time
import datetime
import pandas as pd
import numpy as np
import tensorflow as tf
from termcolor import colored, cprint
from abyss.helpers import metrics_viewer
from abyss.models import create_autoencoder
from abyss.glob import glob
from sklearn.model_selection import train_test_split

In [ ]:
LDs = 15

In [ ]:
bottleneck = 1
epoch = 2

In [ ]:
path_output = glob.PATH_GENO + f"/10_lowest_global_dimensionality/{bottleneck}_dims/"
os.makedirs(path_output, exist_ok = True)

In [ ]:
PATH_input = f"{glob.PATH_GENO}/07_lowest_dimensionality/"

In [ ]:
bins = os.listdir(PATH_input)

In [ ]:
latent_dfs = []
for bin in bins:
    path_bin = PATH_input + bin

    path_lowest = path_bin + f"/{LDs}_dim_latent"

    possible_dims = sorted([int(f.split("_")[3]) for f in os.listdir(path_lowest) if f.split("_")[0] == "reduction"])
    # Select the middle value
    middle_value = possible_dims[len(possible_dims) // 2]
    selected_file = [f for f in os.listdir(path_lowest) if f.split("_")[3] == f"{middle_value}"][0]
    path_file = path_lowest + f"/{selected_file}"
    latent_df = pd.read_pickle(path_file)
    latent_dfs.append(latent_df)

In [ ]:
genos = pd.concat(latent_dfs, axis = 1)

In [ ]:
# Assuming df is your DataFrame
number_of_cols = len(genos.columns)

# Creating new column names
new_column_names = [f'dim{i+1}' for i in range(number_of_cols)]

# Renaming the columns
genos.columns = new_column_names

In [ ]:
genos

In [ ]:
# Activations
activation = 'elu'
depth = 1
    # divide in train - test sets
X = tf.convert_to_tensor(genos, dtype=tf.float32)
# Splitting the data into train and test sets
X_train, X_test = train_test_split(genos, test_size=0.3)  # 70% training and 30% testing

# Convert to TensorFlow tensors
X_train_tensor = tf.convert_to_tensor(X_train, dtype=tf.float32)
X_test_tensor = tf.convert_to_tensor(X_test, dtype=tf.float32)

input_len = int(tf.shape(X_train)[1])

# Fitting the autoencoder
t = time.time()
print(colored(f"Doing dimensionality reduction on {genos.shape[1]} features of {genos.shape[0]} samples using an autoencoder with {epoch} epochs", 'yellow'), colored(' [can take time, be patient]', 'red'))
AE = create_autoencoder(input_len, bottleneck, activation, depth, optimizer = 'Adam', loss = 'mse', output_activation = 'linear')
history = AE.fit(X_train,
            X_train,
            validation_data=(X_test, X_test),
            epochs = epoch,
            verbose = 0)

metric_plt = metrics_viewer(history)
metric_plt.savefig(f"{path_output}/metrics_reduction_concatted_bin_{bin}_bottle_{bottleneck}.png")
#metric_plt.close()

elapsed = round(time.time() - t,2)
time_string = str(datetime.timedelta(seconds=elapsed))
time_array = time_string.split(':')
hours = time_array[0]
minutes = time_array[1]
seconds = round(float(time_array[2]),2)

cprint(f"This took {hours} hours, {minutes} minutes and {seconds} seconds",'red')

# Getting the lower dimensionality hidden representation
hidden_repr = AE.layers[1](X).numpy()

# depending on how many dims in the bottleneck
cols = []
for i2 in range(np.shape(hidden_repr)[1]):
    cols.append(f'dim{i2+1}')
low_dim = pd.DataFrame(hidden_repr, columns = cols)
elapsed = round(elapsed)
low_dim.to_pickle(
    f"{path_output}/reduction_of_ae_{bottleneck}_dims_{elapsed}_seconds_{epoch}_epoch.pkl")